# FINPLE US Price Metrics + Raw Daily Colab

Step 114-2Y extends the existing US yfinance collector. Provider calls run only in this user-operated Colab. Each 100-asset chunk preserves Close, Adj Close, Volume, Dividends, and Stock Splits in the existing `RAW_DAILY_PRICE_COLUMNS` contract and can resume from its checkpoint.


In [ ]:
COLLECTION_REF = "codex/step114-2y-one-click-full-universe-app-preview"
!rm -rf /content/FINPLE
!git clone --depth 1 https://github.com/vip930sw/FINPLE.git /content/FINPLE
%cd /content/FINPLE
!git fetch --depth 1 origin {COLLECTION_REF}
!git checkout {COLLECTION_REF}
!pip -q install yfinance pandas


In [ ]:
from datetime import date, datetime, timezone
from pathlib import Path
import csv, json, subprocess

AS_OF = date.today().isoformat()
RUN_TAG = AS_OF.replace('-', '')
RETRIEVED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
UNIVERSE = Path('src/data/tickers/finple_app_candidates_6000_balanced_v1.csv')
ROOT = Path('/content/finple-step114-2y-us') / RUN_TAG
SMOKE_DIR = ROOT / 'smoke'
CHUNK_DIR = ROOT / 'chunks'
COMBINED_DIR = ROOT / 'combined'
for path in [SMOKE_DIR, CHUNK_DIR, COMBINED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

with UNIVERSE.open('r', encoding='utf-8-sig', newline='') as handle:
    universe_rows = list(csv.DictReader(handle))
US_TOTAL = sum(row['market'] == 'US' for row in universe_rows)
print({'asOf': AS_OF, 'retrievedAt': RETRIEVED_AT, 'usCandidates': US_TOTAL, 'root': str(ROOT)})


## 1. Required 20-asset smoke


In [ ]:
smoke_prefix = SMOKE_DIR / 'us_0000_0020'
subprocess.run([
    'python', 'scripts/build_us_price_metrics_overlay_chunked.py',
    '--input', str(UNIVERSE),
    '--out-runtime', str(smoke_prefix) + '_metrics.csv',
    '--out-audit', str(smoke_prefix) + '_audit.csv',
    '--out-summary', str(smoke_prefix) + '_summary.json',
    '--out-raw', str(smoke_prefix) + '_raw_daily.csv',
    '--as-of', AS_OF, '--start', '0', '--limit', '20',
    '--checkpoint-every', '5', '--retrieved-at', RETRIEVED_AT, '--resume',
], check=True)
with open(str(smoke_prefix) + '_summary.json', encoding='utf-8') as handle:
    smoke_summary = json.load(handle)
assert smoke_summary['processed_count'] == 20
assert smoke_summary['raw_daily_asset_count'] == 20, smoke_summary
print(json.dumps(smoke_summary, indent=2))


## 2. Full US collection in resumable 100-asset chunks


In [ ]:
for start in range(0, US_TOTAL, 100):
    end = min(start + 100, US_TOTAL)
    prefix = CHUNK_DIR / f'us_{start:04d}_{end:04d}'
    subprocess.run([
        'python', 'scripts/build_us_price_metrics_overlay_chunked.py',
        '--input', str(UNIVERSE),
        '--out-runtime', str(prefix) + '_metrics.csv',
        '--out-audit', str(prefix) + '_audit.csv',
        '--out-summary', str(prefix) + '_summary.json',
        '--out-raw', str(prefix) + '_raw_daily.csv',
        '--as-of', AS_OF, '--start', str(start), '--limit', str(end - start),
        '--checkpoint-every', '25', '--retrieved-at', RETRIEVED_AT, '--resume',
    ], check=True)
print('US chunks complete:', len(list(CHUNK_DIR.glob('*_metrics.csv'))))


## 3. Combine and reconcile


In [ ]:
US_METRICS = COMBINED_DIR / 'us_price_metrics_overlay.csv'
US_RAW = COMBINED_DIR / 'us_raw_daily_prices.csv'
US_SUMMARY = COMBINED_DIR / 'us_price_metrics_summary.json'
subprocess.run([
    'python', 'scripts/combine_us_price_metrics_chunks.py',
    '--pattern', str(CHUNK_DIR / '*_metrics.csv'),
    '--out-runtime', str(US_METRICS), '--out-summary', str(US_SUMMARY),
    '--raw-pattern', str(CHUNK_DIR / '*_raw_daily.csv'), '--out-raw', str(US_RAW),
], check=True)
with US_SUMMARY.open(encoding='utf-8') as handle:
    combined_summary = json.load(handle)
assert combined_summary['row_count'] == US_TOTAL
assert combined_summary['rawDailyAssetCount'] <= US_TOTAL
print(json.dumps(combined_summary, indent=2))
print('One-Click inputs:', US_RAW, US_METRICS)


In [ ]:
DOWNLOAD_OUTPUTS = False
if DOWNLOAD_OUTPUTS:
    from google.colab import files
    for target in [US_RAW, US_METRICS, US_SUMMARY]:
        files.download(str(target))
else:
    print('Set DOWNLOAD_OUTPUTS=True after checking reconciliation.')
